In [1]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

c:\Users\Pratikesh\Documents\VS_CODE\LLM_RAG_practices\RAG_Langchain\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
#setup
directory =  "dbv2/chroma_db"
embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small")

In [11]:
db = Chroma(persist_directory=directory,embedding_function=embedding_model,collection_configuration={"hnsw:space": "cosine"})

#query to test
Query = "What are Transmission of disease?"

print(f"Query  => {Query}\n")

Query  => What are Transmission of disease?



# METHOD 1 :BASIC Similarity Search

In [12]:
print("=== METHOD 1: Similarity Search (k=3) ===")

ret = db.as_retriever(search_kwargs = {"k":3})

docs = ret.invoke(Query)



for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(f"{doc.page_content}\n")


=== METHOD 1: Similarity Search (k=3) ===
Document 1:
DISEASE AND DISEASE TRANSMISSION

Chapter 2

Disease and disease transmission

An enormous variety of organisms exist, including some which can survive and even develop in the body of people or animals. If the organism can cause infection, it is an infectious agent. In this manual infectious agents which cause infection and illness are called pathogens. Diseases caused by pathogens, or the toxins they produce, are communicable or infectious diseases (45). In this manual these will be called disease and infection.

This chapter presents the transmission cycle of disease with its different elements, and categorises the different infections related to WES.

Document 2:
2.1 Introduction to the transmission cycle of disease

To be able to persist or live on, pathogens must be able to leave an infected host, survive transmission in the environment, enter a susceptible person or animal, and develop and/or multiply in the newly infected hos

# METHOD 2: Similarity with Score Threshold
##Only returns documents above a certain similarity score

In [13]:
print("\n=== METHOD 2: Similarity with Score Threshold ===")
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.3  # Only return docs with similarity >= 0.3
    }
)

docs = retriever.invoke(Query)
print(f"Retrieved {len(docs)} documents (threshold: 0.3):\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(f"{doc.page_content}\n")



=== METHOD 2: Similarity with Score Threshold ===
Retrieved 3 documents (threshold: 0.3):

Document 1:
DISEASE AND DISEASE TRANSMISSION

Chapter 2

Disease and disease transmission

An enormous variety of organisms exist, including some which can survive and even develop in the body of people or animals. If the organism can cause infection, it is an infectious agent. In this manual infectious agents which cause infection and illness are called pathogens. Diseases caused by pathogens, or the toxins they produce, are communicable or infectious diseases (45). In this manual these will be called disease and infection.

This chapter presents the transmission cycle of disease with its different elements, and categorises the different infections related to WES.

Document 2:
2.1 Introduction to the transmission cycle of disease

To be able to persist or live on, pathogens must be able to leave an infected host, survive transmission in the environment, enter a susceptible person or animal, and

# METHOD 3: Maximum Marginal Relevance (MMR)
##Balances relevance and diversity - avoids redundant results

In [14]:
print("\n=== METHOD 3: Maximum Marginal Relevance (MMR) ===")
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,           # Final number of docs
        "fetch_k": 10,    # Initial pool to select from
        "lambda_mult": 0.5  # 0=max diversity, 1=max relevance
    }
)

docs = retriever.invoke(Query)
print(f"Retrieved {len(docs)} documents (λ=0.5):\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(f"{doc.page_content}\n")


=== METHOD 3: Maximum Marginal Relevance (MMR) ===
Retrieved 3 documents (λ=0.5):

Document 1:
DISEASE AND DISEASE TRANSMISSION

Chapter 2

Disease and disease transmission

An enormous variety of organisms exist, including some which can survive and even develop in the body of people or animals. If the organism can cause infection, it is an infectious agent. In this manual infectious agents which cause infection and illness are called pathogens. Diseases caused by pathogens, or the toxins they produce, are communicable or infectious diseases (45). In this manual these will be called disease and infection.

This chapter presents the transmission cycle of disease with its different elements, and categorises the different infections related to WES.

Document 2:
2.4.2.5 Vector-borne diseases

These infections are transmitted by vectors. Vectors are arthropods (insects, ticks, or mites) which can transmit infections from host to future host (73). The pathogen exists in the blood or skin o